In [0]:
import numpy as np
import pandas as pd

def split_valid_invalid_orders_pdf(pdf: pd.DataFrame):
    work = pdf.copy()

    work["order_id"] = pd.to_numeric(work["order_id"], errors="coerce")
    work["customer_id"] = pd.to_numeric(work["customer_id"], errors="coerce")
    work["amount"] = pd.to_numeric(work["amount"], errors="coerce")
    work["order_date"] = pd.to_datetime(work["order_date"], errors="coerce").dt.date

    invalid_mask = (
        work["order_id"].isna() |
        work["customer_id"].isna() |
        work["amount"].isna() |
        (work["amount"] < 0) |
        work["order_date"].isna()
    )

    invalid = work.loc[invalid_mask].copy()

    conditions = [
        invalid["order_id"].isna(),
        invalid["customer_id"].isna(),
        invalid["amount"].isna(),
        invalid["amount"] < 0,
        invalid["order_date"].isna()
    ]
    choices = [
        "NULL_ORDER_ID",
        "NULL_CUSTOMER_ID",
        "NULL_AMOUNT",
        "NEGATIVE_AMOUNT",
        "NULL_ORDER_DATE"
    ]

    invalid["reject_reason"] = np.select(conditions, choices, default="UNKNOWN")
    valid = work.loc[~invalid_mask].copy()

    return valid, invalid

def dedupe_customers_pdf(pdf: pd.DataFrame):
    work = pdf.copy()
    work["updated_at"] = pd.to_datetime(work["updated_at"], errors="coerce")
    work["_ingestion_ts"] = pd.to_datetime(work["_ingestion_ts"], errors="coerce")

    work = work.sort_values(
        by=["customer_id", "updated_at", "_ingestion_ts"],
        ascending=[True, False, False]
    )

    return work.drop_duplicates(subset=["customer_id"], keep="first").copy()

def compute_reject_rate(total_rows: int, reject_rows: int) -> float:
    if total_rows == 0:
        return 0.0
    return reject_rows / total_rows

def build_gold_daily_sales_pdf(orders_pdf: pd.DataFrame):
    work = orders_pdf.copy()
    work["order_date"] = pd.to_datetime(work["order_date"], errors="coerce").dt.date
    work["amount"] = pd.to_numeric(work["amount"], errors="coerce")

    out = (
        work.groupby("order_date", as_index=False)
        .agg(
            total_orders=("order_id", "count"),
            total_revenue=("amount", "sum"),
            avg_ticket=("amount", "mean")
        )
        .sort_values("order_date")
    )

    out["total_revenue"] = out["total_revenue"].round(2)
    out["avg_ticket"] = out["avg_ticket"].round(2)
    return out

def build_gold_sales_by_country_pdf(orders_pdf: pd.DataFrame, customers_pdf: pd.DataFrame):
    orders = orders_pdf.copy()
    customers = customers_pdf.copy()

    orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce").dt.date
    orders["amount"] = pd.to_numeric(orders["amount"], errors="coerce")

    merged = orders.merge(
        customers[["customer_id", "country"]],
        on="customer_id",
        how="left"
    )

    out = (
        merged.groupby(["order_date", "country"], as_index=False)
        .agg(
            total_orders=("order_id", "count"),
            total_revenue=("amount", "sum")
        )
        .sort_values(["order_date", "country"])
    )

    out["total_revenue"] = out["total_revenue"].round(2)
    return out

def detect_revenue_drop(gold_daily_pdf: pd.DataFrame, current_run_date: str, min_history_days: int = 3):
    work = gold_daily_pdf.copy()
    work["order_date"] = pd.to_datetime(work["order_date"], errors="coerce").dt.date
    current_date = pd.to_datetime(current_run_date).date()

    current_row = work.loc[work["order_date"] == current_date]
    previous_rows = work.loc[work["order_date"] < current_date].sort_values("order_date").tail(7)

    if current_row.empty or len(previous_rows) < min_history_days:
        return None

    current_revenue = float(current_row["total_revenue"].iloc[0])
    avg_prev_7 = float(previous_rows["total_revenue"].mean())

    if avg_prev_7 <= 0:
        return None

    ratio = current_revenue / avg_prev_7

    if ratio < 0.60:
        return {
            "current_run_date": str(current_date),
            "current_revenue": round(current_revenue, 2),
            "avg_prev_7": round(avg_prev_7, 2),
            "ratio": round(ratio, 4)
        }

    return None